# 🧬 ToxFam — Toxin Family Prediction

Predict the **top-k toxin protein families** for your sequences with the trained ToxFam models.

**What you need:** a `FASTA` file *or* precomputed ProtT5 embeddings (`.h5`).

**How to use:**
1. **Runtime → Change runtime type → GPU** (recommended — FASTA embedding is slow on CPU).
2. **Runtime → Run all** (or run each cell top-to-bottom).
3. In the panel at the bottom: choose your input type, upload your file, pick the model, and click **Run**.
4. View the table of top predictions and click **Download CSV**.

**Models**
- **standard** — predicts the family from the ProtT5 embedding alone. Works for any input.
- **combined** — also uses organism taxonomy (more context). Only available when an NCBI
  taxon ID is known for *every* sequence — supplied via `OX=<taxid>` in UniProt FASTA headers,
  or an uploaded `identifier,taxid` CSV.


In [ ]:
#@title 1. Setup — install ToxFam and check GPU { display-mode: "form" }
import subprocess, sys

print("Installing ToxFam (takes ~1-2 min on first run)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/Sisistern123/ToxFam.git"],
    check=True,
)

import torch
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

if torch.cuda.is_available():
    print("Device: GPU ✅", torch.cuda.get_device_name(0))
else:
    print("Device: CPU ⚠️  FASTA embedding will be slow.")
    print("   Tip: Runtime → Change runtime type → GPU, then re-run.")
print("Setup complete.")


In [ ]:
#@title 2. Download the trained models { display-mode: "form" }
import urllib.request, zipfile, os

URL = "https://github.com/Sisistern123/ToxFam/releases/download/models-v2/models.zip"
print("Downloading models.zip ...")
urllib.request.urlretrieve(URL, "models.zip")
with zipfile.ZipFile("models.zip") as zf:
    zf.extractall(".")
ready = [d for d in ("standard_run", "combined_run") if os.path.isdir(d)]
print("Models ready:", ready)


In [ ]:
#@title 3. Load helper functions { display-mode: "form" }
import os, re
import h5py, torch, pandas as pd

from toxfam.data._fasta import read_fasta_as_dict
from toxfam.data.embedding import generate_embeddings
from toxfam.model.inference import load_calibrated_model

OX_RE = re.compile(r"OX=(\d+)")


def _clean_id(token):
    """Match read_fasta_as_dict identifier cleaning ('/' and '.' -> '_')."""
    return token.replace("/", "_").replace(".", "_")


def taxids_from_headers(fasta_path):
    """{identifier: taxid} for FASTA headers that carry an OX=<taxid> tag."""
    out = {}
    with open(fasta_path) as fh:
        for line in fh:
            if line.startswith(">"):
                head = line[1:].strip()
                ident = _clean_id(head.split()[0])
                m = OX_RE.search(head)
                if m:
                    out[ident] = int(m.group(1))
    return out


def taxids_from_csv(csv_path):
    """{identifier: taxid} from a CSV with 'identifier' and 'taxid' columns."""
    df = pd.read_csv(csv_path)
    cols = {c.lower().strip(): c for c in df.columns}
    id_col = cols.get("identifier")
    tax_col = cols.get("taxid") or cols.get("organism (id)") or cols.get("organism id")
    if id_col is None or tax_col is None:
        raise ValueError("taxid CSV must have columns 'identifier' and 'taxid'")
    out = {}
    for _, r in df.iterrows():
        try:
            out[_clean_id(str(r[id_col]))] = int(r[tax_col])
        except (ValueError, TypeError):
            pass
    return out


def h5_identifiers(h5_path):
    with h5py.File(h5_path, "r") as f:
        return list(f.keys())


def build_taxonomy_h5(emb_h5, identifiers, taxids, out_h5):
    """Build multi-hot taxonomy vectors via the package pipeline (downloads NCBI taxdump once)."""
    from toxfam.data.taxonomy import run_multi_hot_taxonomy_pipeline

    csv_path = out_h5 + ".csv"
    pd.DataFrame(
        {"identifier": identifiers, "Organism (ID)": [taxids[i] for i in identifiers]}
    ).to_csv(csv_path, index=False)
    run_multi_hot_taxonomy_pipeline(csv_path, emb_h5, out_h5, id_col="identifier")


def predict_topk(model_dir, emb_h5, identifiers, k=3, tax_h5=None):
    """Return a DataFrame: identifier, rank1..k, conf1..k."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, cfg, idx2lab = load_calibrated_model(model_dir, device=device)
    is_multi = cfg.architecture == "MultiInputMLP"

    with h5py.File(emb_h5, "r") as f:
        emb = torch.stack(
            [torch.tensor(f[i][:], dtype=torch.float32) for i in identifiers]
        )
    tax = None
    if is_multi:
        with h5py.File(tax_h5, "r") as tf:
            tax = torch.stack(
                [torch.tensor(tf[i][:], dtype=torch.float32) for i in identifiers]
            )

    rows, bs = [], 512
    with torch.no_grad():
        for s in range(0, len(emb), bs):
            b = emb[s : s + bs].to(device)
            logits = model(b, tax[s : s + bs].to(device)) if is_multi else model(b)
            probs = torch.softmax(logits, dim=1)
            kk = min(k, probs.shape[1])
            confs, idxs = torch.topk(probs, k=kk, dim=1)
            for r in range(b.shape[0]):
                row = {"identifier": identifiers[s + r]}
                for j in range(kk):
                    row[f"rank{j + 1}"] = idx2lab[idxs[r, j].item()]
                    row[f"conf{j + 1}"] = round(confs[r, j].item(), 4)
                rows.append(row)
    return pd.DataFrame(rows)


print("Helpers loaded.")


In [ ]:
#@title 4. Predict { display-mode: "form" }
# Colab's ipywidgets FileUpload often returns an empty value, so file picking
# uses google.colab.files.upload() (reliable); ipywidgets handles the rest.
import ipywidgets as W
from IPython.display import display
from google.colab import files as colab_files

input_type = W.RadioButtons(
    options=["FASTA", "Embeddings (.h5)"], description="Input:",
    style={"description_width": "initial"},
)
up_in_btn = W.Button(description="Upload input file", icon="upload")
up_tax_btn = W.Button(description="Upload taxid CSV (optional)", icon="upload")
model_dd = W.Dropdown(options=["standard"], description="Model:")
topk = W.IntSlider(value=3, min=1, max=5, description="Top-k:")
run_btn = W.Button(description="Run", button_style="success", icon="play")
dl_btn = W.Button(description="Download CSV", icon="download")
status = W.HTML()
out = W.Output()

state = {"input_path": None, "taxid_path": None, "results": None}


def _pick_file(default_name):
    """Open Colab's native uploader; return the saved /content path (or None)."""
    uploaded = colab_files.upload()  # {filename: bytes}
    if not uploaded:
        return None
    name = next(iter(uploaded))
    path = "/content/" + name
    with open(path, "wb") as f:
        f.write(uploaded[name])
    return path


def _gather():
    """Return (input_type, input_path, identifiers, {id: taxid}) from state."""
    itype = input_type.value
    ipath = state["input_path"]
    ids, taxids = [], {}
    if ipath:
        if itype == "FASTA":
            ids = list(read_fasta_as_dict(ipath).keys())
            taxids.update(taxids_from_headers(ipath))
        else:
            ids = h5_identifiers(ipath)
    if state["taxid_path"]:
        for k, v in taxids_from_csv(state["taxid_path"]).items():
            taxids.setdefault(k, v)
    return itype, ipath, ids, taxids


def _refresh(change=None):
    try:
        itype, ipath, ids, taxids = _gather()
    except Exception as e:
        status.value = f"<span style='color:#b00'>Input problem: {e}</span>"
        return
    if not ipath:
        model_dd.options = ["standard"]
        status.value = "<i>Click 'Upload input file' to begin.</i>"
        return
    fname = ipath.split("/")[-1]
    complete = bool(ids) and all(i in taxids for i in ids)
    if complete:
        model_dd.options = ["standard", "combined"]
        status.value = (
            f"<span style='color:#070'>✓ <b>{fname}</b>: {len(ids)} sequences, "
            f"taxon IDs found for all — <b>combined</b> model available.</span>"
        )
    else:
        model_dd.options = ["standard"]
        missing = [i for i in ids if i not in taxids]
        why = "no taxon IDs provided" if not taxids else f"{len(missing)} sequence(s) missing taxon IDs"
        status.value = (
            f"<span style='color:#a60'>✓ <b>{fname}</b>: {len(ids)} sequences — {why}. "
            f"Only the <b>standard</b> model is available. Add OX= headers or a taxid CSV "
            f"to enable combined.</span>"
        )


def _on_upload_input(_):
    with out:
        out.clear_output()
        print("Choose your input file ...")
        p = _pick_file("input.dat")
    if p:
        state["input_path"] = p
        out.clear_output()
    _refresh()


def _on_upload_tax(_):
    with out:
        out.clear_output()
        print("Choose your taxid CSV ...")
        p = _pick_file("taxids.csv")
    if p:
        state["taxid_path"] = p
        out.clear_output()
    _refresh()


input_type.observe(_refresh, names="value")
up_in_btn.on_click(_on_upload_input)
up_tax_btn.on_click(_on_upload_tax)


def _run(_):
    out.clear_output()
    with out:
        try:
            itype, ipath, ids, taxids = _gather()
            if not ipath:
                print("Please click 'Upload input file' first.")
                return
            if not ids:
                print("No sequences found in the input.")
                return
            model = model_dd.value
            if model == "combined" and not all(i in taxids for i in ids):
                print("Combined needs a taxon ID for every sequence. Use standard or add taxids.")
                return

            if itype == "FASTA":
                print(f"Embedding {len(ids)} sequences (first run downloads ProtT5 ~2.5 GB)...")
                emb_h5 = "/content/_emb.h5"
                if os.path.exists(emb_h5):
                    os.remove(emb_h5)
                generate_embeddings(ipath, emb_h5)
            else:
                emb_h5 = ipath

            tax_h5 = None
            if model == "combined":
                print("Building taxonomy vectors (first run downloads NCBI taxdump ~60 MB)...")
                tax_h5 = "/content/_tax.h5"
                if os.path.exists(tax_h5):
                    os.remove(tax_h5)
                build_taxonomy_h5(emb_h5, ids, taxids, tax_h5)

            print(f"Running {model} model on {len(ids)} sequences ...")
            df = predict_topk(f"/content/{model}_run", emb_h5, ids, k=topk.value, tax_h5=tax_h5)
            state["results"] = df
            df.to_csv("/content/toxfam_predictions.csv", index=False)
            print("Done.\n")
            display(df)
            print("\nSaved to toxfam_predictions.csv — click 'Download CSV' above.")
        except Exception:
            import traceback
            traceback.print_exc()


def _dl(_):
    with out:
        if state["results"] is None:
            print("Run a prediction first.")
        else:
            colab_files.download("/content/toxfam_predictions.csv")


run_btn.on_click(_run)
dl_btn.on_click(_dl)

display(
    W.VBox([
        W.HTML("<h3>ToxFam — top-k toxin family prediction</h3>"),
        input_type,
        W.HBox([up_in_btn, up_tax_btn]),
        model_dd,
        topk,
        W.HBox([run_btn, dl_btn]),
        status,
        out,
    ])
)
_refresh()
